# 04. Comparaciones finales y verificación

Este cuaderno recalcula el análisis completo y contrasta sus 35 CSV con las referencias congeladas. Después muestra las comparaciones de calidad, su soporte y los descriptivos de generación desde la ejecución nueva.

Run All tarda más que los cuadernos de inspección porque repite las estimaciones bootstrap. No llama a modelos ni cambia los documentos.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# Jupyter puede abrir el cuaderno desde la raíz técnica o desde notebooks/.
HERE = Path.cwd().resolve()
SUPPORT = next(p / 'notebooks' for p in (HERE, *HERE.parents)
               if (p / 'notebooks/notebook_support.py').is_file())
sys.path.insert(0, str(SUPPORT))
from notebook_support import find_root, read_jsonl, new_run, run_script, compare_csv

ROOT = find_root()
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 90)

## Ejecutar el reanálisis completo

El programa común encadena validación, recuperación, explicitud, comparación de generadores y comparación de recuperadores dentro de cada generador. Utiliza un identificador nuevo en cada ejecución.

In [ ]:
LOGS = new_run('comparacion_final')
run_id = 'notebook_final_' + LOGS.name.rsplit('_', 1)[-1]
report = json.loads(run_script('scripts/reproduce_offline.py', '--run-id', run_id, log_dir=LOGS))
RUN = ROOT / report['output']
assert report['compared_csv_count'] == 35
assert report['new_inference'] is False and report['documents_rendered'] is False
checks = [{'etapa': stage, 'archivo': name, 'identico': True}
          for stage, files in report['csv_sha256_equal_to_frozen'].items() for name in files]
assert len(checks) == 35
display(pd.DataFrame(checks))
print('Salidas:', RUN.relative_to(ROOT))

In [ ]:
validation = json.loads((RUN / 'validation.json').read_text())
assert validation['table_count'] == 11 and validation['numeric_cells'] == 389
display(pd.DataFrame({'comprobacion': ['Tablas numericas', 'Valores numericos', 'CSV reproducidos'],
                      'n': [validation['table_count'], validation['numeric_cells'], report['compared_csv_count']]}))
display(pd.Series(validation['pdf'], name='valor').to_frame())

## Recuperadores dentro de cada generador

El análisis intersecta las formulaciones disponibles entre las cuatro estrategias para cada generador y criterio. Primero promedia dentro de pregunta base y después entre bases. Así evita que una estrategia parezca mejor por haber sido evaluada sobre preguntas distintas.

In [ ]:
main = pd.read_csv(RUN / 'final/main_table.csv', na_values=['', 'NA', 'na'])
assert len(main) == 12
display(main[['generator_model', 'retrieval_condition', 'faithfulness_mean',
              'factual_correctness_mean', 'behavior_ok_mean', 'citations_ok_mean']])

Las columnas de disponibilidad explican cuántas observaciones quedan fuera del soporte común. Las abstenciones se mantienen en los criterios de conducta y citas; las métricas de contenido se restringen a los casos previstos en el protocolo.

In [ ]:
availability = pd.read_csv(RUN / 'final/availability.csv', na_values=['', 'NA', 'na'])
assert len(availability) == 48
display(availability[['generator_model', 'retrieval_condition', 'criterion',
                      'n_accepted', 'n_in_scope', 'n_unavailable', 'n_common', 'n_numeric_excluded']])

Cada diferencia de la tabla siguiente es **estrategia de destino menos estrategia de origen**. Los intervalos bootstrap son puntuales, descriptivos y sin ajuste por la familia de comparaciones. Un intervalo que incluye cero no demuestra equivalencia.

In [ ]:
contrasts = pd.read_csv(RUN / 'final/paired_contrasts.csv', na_values=['', 'NA', 'na'])
assert len(contrasts) == 72
display(contrasts[['generator_model', 'criterion', 'from_condition', 'to_condition',
                   'diff', 'ic95_lo', 'ic95_hi', 'n_cells', 'n_base']])

## Comparación de generadores

La comparación entre generadores utiliza su propio soporte emparejado. La tabla permite examinar los criterios de contenido sin mezclar sus denominadores con los de la comparación anterior. El índice externo de capacidad no se utiliza aquí como una medición independiente de calidad.

In [ ]:
generators = pd.read_csv(RUN / 'capacity/h6_niveles_common.csv', na_values=['', 'NA', 'na'])
display(generators.loc[generators['scope'] == 'primary_content',
    ['generator_model', 'faithfulness', 'faithfulness_n_base', 'faithfulness_n_cells',
     'factual_correctness', 'factual_correctness_n_base', 'factual_correctness_n_cells']])
display(pd.read_csv(RUN / 'capacity/h5_comparaciones_common.csv', na_values=['', 'NA', 'na']))

## Tiempos y cobertura

Los tiempos proceden de la generación registrada e incluyen los intentos correctivos acumulados. No incluyen recuperación ni evaluación, y no constituyen un benchmark end-to-end controlado. Los fallos se muestran aparte de los tiempos medios de las respuestas aceptadas.

In [ ]:
operations = pd.read_csv(RUN / 'final/operational.csv', na_values=['', 'NA', 'na'])
display(operations)
cells = pd.read_csv(RUN / 'final/operational_cells.csv', na_values=['', 'NA', 'na'])
assert len(cells) == 624 and cells['status'].value_counts().to_dict() == {'accepted': 604, 'failed': 20}
display(cells.groupby(['generator_model', 'status'])
        .agg(unidades=('qid', 'size'), intentos=('generation_attempts', 'sum'),
             segundos_mediana=('latency_s', 'median'), segundos_maximo=('latency_s', 'max')))

## Figuras de las memorias, opcionales

La opción siguiente ejecuta el código R literal de las fuentes y guarda diez SVG en otro destino nuevo. Requiere R con dplyr, ggplot2, jsonlite, knitr y tibble. Se deja desactivada para que el reanálisis numérico pueda hacerse solo con el entorno Python.

In [ ]:
REGENERAR_FIGURAS = False
if REGENERAR_FIGURAS:
    figures = json.loads(run_script('scripts/reproduce_figures.py', '--run-id', run_id, log_dir=LOGS))
    assert figures['r_chunks_executed'] == 11 and figures['svg_count'] == 10
    display(pd.DataFrame({'figura': figures['svg_files']}))

La reproducción queda comprobada cuando los 35 CSV coinciden con las referencias y los validadores aceptan las tablas y los documentos. Eso acredita el reanálisis de las fuentes incluidas; no una nueva generación de respuestas ni una repetición de la revisión humana.